Imports library and data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import joblib
import json
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent 
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

customer_path = str(RAW_DIR / "customer_info.parquet")
usage_path = str(RAW_DIR / "usage.parquet")
calls_path = str(RAW_DIR / "calls.csv")
cease_path = str(RAW_DIR / "cease.csv")

RAW_DIR

WindowsPath('c:/Users/icons/Documents/UKTelecomCeasePrediction/data/raw')

File sizes check

In [2]:
for p in RAW_DIR.iterdir():
    print(f"{p.name:22s} {p.stat().st_size/1e9:8.3f} GB")

calls.csv                 0.059 GB
cease.csv                 0.018 GB
customer_info.parquet     0.270 GB
usage.parquet             3.094 GB


Load tables

In [3]:
cease = pd.read_csv(
    cease_path,
    parse_dates=["cease_placed_date", "cease_completed_date"],
)

calls = pd.read_csv(
    calls_path,
    usecols=["unique_customer_identifier", "event_date", "call_type",
             "talk_time_seconds", "hold_time_seconds"],
    parse_dates=["event_date"],
)

print("cease:", cease.shape, "calls:", calls.shape)
display(cease.head(3))
display(calls.head(3))

cease: (146363, 5) calls: (628437, 5)


,unique_customer_identifier,cease_placed_date,cease_completed_date,reason_description,reason_description_insight
0,03b1c584533a86d067dd51bbca242db2b55b692f10d325...,2023-08-03,2023-09-04,Competitor Deals - No longer required,CompetitorDeals
1,97a7bdce317de91a32636e6675bbb2e5b25573308ef7bb...,2023-08-03,2023-09-04,Cease,VagueReason
2,c5049a1aedc36d7d7379c2c2144972b099521e6614cf8c...,2023-08-03,2023-09-05,Competitor Deals - No longer required,CompetitorDeals


,unique_customer_identifier,event_date,call_type,talk_time_seconds,hold_time_seconds
0,aae0258b41e6e88365d7d5ce648ea69d837602b4bb419e...,2023-02-22,Loyalty,627.0,235.0
1,15f9f6fc1872bbf6963a84de253d600e5d18d75d7784ce...,2023-03-16,Tech,267.0,293.0
2,c18d59888cb050a5694d1e613a277d79b4a3083bd1b813...,2023-02-22,Loyalty,689.0,0.0


Checking datatypes and missing data

In [4]:
calls["call_type"] = calls["call_type"].astype(str).fillna("Unknown").str.strip()
cease["reason_description"] = cease.get("reason_description", pd.Series(["Unknown"]*len(cease))).astype(str).str.strip()

print("Top call types:")
display(calls["call_type"].value_counts().head(20))

Top call types:


call_type
Tech                            171942
CS&B                            167117
Loyalty                         144269
Customer Finance                 71105
FTTP                             43926
nan                              17948
Other                             5385
Order Management                  4422
Complaints                        1504
TTB - Sales                        369
TTB - Customer Services            313
TTB - Credit Control                80
TTBD - Customer Experience          35
TTB - Technical Support             13
TTB - Service Implementation         5
TTB - Fulfilment                     4
Name: count, dtype: int64

Print date ranges

In [5]:
print("cease placed:", cease["cease_placed_date"].min(), "->", cease["cease_placed_date"].max())
print("calls event :", calls["event_date"].min(), "->", calls["event_date"].max())

cease placed: 2022-08-01 00:00:00 -> 2024-09-01 00:00:00
calls event : 2022-08-01 00:00:00 -> 2024-09-01 00:00:00


DuckDB connect for big tables

In [6]:
con = duckdb.connect()

customer_cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{customer_path}')").fetchdf()
usage_cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{usage_path}')").fetchdf()

print("customer_info cols:", len(customer_cols))
print("usage cols:", len(usage_cols))

display(customer_cols[["column_name","column_type"]])
display(usage_cols[["column_name","column_type"]])

customer_info cols: 12
usage cols: 4


,column_name,column_type
0,unique_customer_identifier,VARCHAR
1,datevalue,DATE
2,contract_status,VARCHAR
3,contract_dd_cancels,BIGINT
4,dd_cancel_60_day,INTEGER
5,ooc_days,INTEGER
6,technology,VARCHAR
7,speed,INTEGER
8,line_speed,DOUBLE
9,sales_channel,VARCHAR


,column_name,column_type
0,unique_customer_identifier,VARCHAR
1,calendar_date,DATE
2,usage_download_mbs,VARCHAR
3,usage_upload_mbs,VARCHAR


Validate Required columns exist

In [7]:
required_customer = {"unique_customer_identifier", "datevalue",
                     "contract_status","contract_dd_cancels","dd_cancel_60_day","ooc_days",
                     "technology","speed","line_speed","sales_channel","crm_package_name","tenure_days"}

required_usage = {"unique_customer_identifier","calendar_date","usage_download_mbs","usage_upload_mbs"}

missing_cust = required_customer - set(customer_cols["column_name"])
missing_usage = required_usage - set(usage_cols["column_name"])

print("Missing customer columns:", missing_cust)
print("Missing usage columns:", missing_usage)

assert not missing_cust, "customer_info missing required columns"
assert not missing_usage, "usage missing required columns"

Missing customer columns: set()
Missing usage columns: set()


Register pandas tables into DuckDB

In [8]:
con.register("calls_pd", calls)
con.register("cease_pd", cease)

Defining target and feature windows

Predicting if a Customer will place a cease within the nexr 30 days

Set Windows

In [9]:
LOOKBACK_DAYS = 30
HORIZON_DAYS = 30

Determine top call types

In [10]:
TOP_CALL_TYPES = (
    calls["call_type"]
    .value_counts()
    .head(6)  # keep small & meaningful
    .index
    .tolist()
)
TOP_CALL_TYPES

['Tech', 'CS&B', 'Loyalty', 'Customer Finance', 'FTTP', 'nan']

Build modeling table

In [11]:
modeling_path = str(INTERIM_DIR / "modeling_table.parquet")

# Build dynamic SQL for top call type counts
call_type_sum_sql = ",\n".join([
    f"SUM(CASE WHEN ca.call_type = '{ct}' THEN 1 ELSE 0 END) AS calls_{ct.replace(' ', '_').replace('&','and').replace('/','_')}_30d"
    for ct in TOP_CALL_TYPES
])

con.execute(f"""
COPY (
WITH base AS (
    SELECT
        unique_customer_identifier,
        CAST(datevalue AS DATE) AS snapshot_date,
        TRIM(contract_status) AS contract_status,
        contract_dd_cancels,
        dd_cancel_60_day,
        ooc_days,
        TRIM(technology) AS technology,
        speed,
        line_speed,
        TRIM(sales_channel) AS sales_channel,
        TRIM(crm_package_name) AS crm_package_name,
        tenure_days
    FROM read_parquet('{customer_path}')
),

labels AS (
    SELECT
        b.unique_customer_identifier,
        b.snapshot_date,
        CASE WHEN EXISTS (
            SELECT 1 FROM cease_pd c
            WHERE c.unique_customer_identifier = b.unique_customer_identifier
              AND CAST(c.cease_placed_date AS DATE) > b.snapshot_date
              AND CAST(c.cease_placed_date AS DATE) <= b.snapshot_date + INTERVAL '{HORIZON_DAYS} days'
        ) THEN 1 ELSE 0 END AS target_30d
    FROM base b
),

calls_agg AS (
    SELECT
        b.unique_customer_identifier,
        b.snapshot_date,
        COUNT(ca.event_date) AS calls_30d,
        COALESCE(AVG(ca.talk_time_seconds), 0) AS avg_talk_time_30d,
        COALESCE(AVG(ca.hold_time_seconds), 0) AS avg_hold_time_30d,
        {call_type_sum_sql}
    FROM base b
    LEFT JOIN calls_pd ca
      ON ca.unique_customer_identifier = b.unique_customer_identifier
     AND CAST(ca.event_date AS DATE) > b.snapshot_date - INTERVAL '{LOOKBACK_DAYS} days'
     AND CAST(ca.event_date AS DATE) <= b.snapshot_date
    GROUP BY 1,2
),

usage_agg AS (
    SELECT
        b.unique_customer_identifier,
        b.snapshot_date,
        COALESCE(AVG(TRY_CAST(u.usage_download_mbs AS DOUBLE)), 0) AS avg_download_30d,
        COALESCE(AVG(TRY_CAST(u.usage_upload_mbs   AS DOUBLE)), 0) AS avg_upload_30d,
        COALESCE(SUM(TRY_CAST(u.usage_download_mbs AS DOUBLE)), 0) AS sum_download_30d,
        COALESCE(SUM(TRY_CAST(u.usage_upload_mbs   AS DOUBLE)), 0) AS sum_upload_30d
    FROM base b
    LEFT JOIN read_parquet('{usage_path}') u
      ON u.unique_customer_identifier = b.unique_customer_identifier
     AND CAST(u.calendar_date AS DATE) > b.snapshot_date - INTERVAL '{LOOKBACK_DAYS} days'
     AND CAST(u.calendar_date AS DATE) <= b.snapshot_date
    GROUP BY 1,2
)

SELECT
    b.*,
    l.target_30d,
    ca.calls_30d, ca.avg_talk_time_30d, ca.avg_hold_time_30d,
    {", ".join([f"ca.calls_{ct.replace(' ', '_').replace('&','and').replace('/','_')}_30d" for ct in TOP_CALL_TYPES])},
    ua.avg_download_30d, ua.avg_upload_30d, ua.sum_download_30d, ua.sum_upload_30d
FROM base b
JOIN labels l
  ON l.unique_customer_identifier = b.unique_customer_identifier
 AND l.snapshot_date = b.snapshot_date
LEFT JOIN calls_agg ca
  ON ca.unique_customer_identifier = b.unique_customer_identifier
 AND ca.snapshot_date = b.snapshot_date
LEFT JOIN usage_agg ua
  ON ua.unique_customer_identifier = b.unique_customer_identifier
 AND ua.snapshot_date = b.snapshot_date
) TO '{modeling_path}' (FORMAT PARQUET);
""")

print("Saved:", modeling_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: c:\Users\icons\Documents\UKTelecomCeasePrediction\data\interim\modeling_table.parquet


Check duplicates

In [12]:
dup_summary = con.execute(f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT unique_customer_identifier || '|' || CAST(snapshot_date AS VARCHAR)) AS n_unique_keys,
  COUNT(*) - COUNT(DISTINCT unique_customer_identifier || '|' || CAST(snapshot_date AS VARCHAR)) AS n_duplicates
FROM read_parquet('{modeling_path}')
""").fetchdf()

dup_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_unique_keys,n_duplicates
0,3571174,3532720,38454


Deduplicate

In [13]:
dedup_path = str(INTERIM_DIR / "modeling_table_dedup.parquet")

con.execute(f"""
COPY (
  SELECT *
  FROM (
    SELECT
      *,
      (
        CASE WHEN contract_status IS NULL THEN 1 ELSE 0 END +
        CASE WHEN technology IS NULL THEN 1 ELSE 0 END +
        CASE WHEN sales_channel IS NULL THEN 1 ELSE 0 END +
        CASE WHEN crm_package_name IS NULL THEN 1 ELSE 0 END
      ) AS null_score,
      ROW_NUMBER() OVER (
        PARTITION BY unique_customer_identifier, snapshot_date
        ORDER BY null_score ASC
      ) AS rn
    FROM read_parquet('{modeling_path}')
  )
  WHERE rn = 1
) TO '{dedup_path}' (FORMAT PARQUET);
""")

dedup_check = con.execute(f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT unique_customer_identifier || '|' || CAST(snapshot_date AS VARCHAR)) AS n_unique_keys,
  COUNT(*) - COUNT(DISTINCT unique_customer_identifier || '|' || CAST(snapshot_date AS VARCHAR)) AS n_duplicates
FROM read_parquet('{dedup_path}')
""").fetchdf()

dedup_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_unique_keys,n_duplicates
0,3532720,3532720,0


Create time splits (train/val/test)

In [14]:
train_path = str(INTERIM_DIR / "train.parquet")
val_path   = str(INTERIM_DIR / "val.parquet")
test_path  = str(INTERIM_DIR / "test.parquet")

con.execute(f"""
COPY (
  SELECT * FROM read_parquet('{dedup_path}')
  WHERE snapshot_date <= DATE '2024-03-01'
) TO '{train_path}' (FORMAT PARQUET);
""")

con.execute(f"""
COPY (
  SELECT * FROM read_parquet('{dedup_path}')
  WHERE snapshot_date >= DATE '2024-04-01' AND snapshot_date <= DATE '2024-06-01'
) TO '{val_path}' (FORMAT PARQUET);
""")

con.execute(f"""
COPY (
  SELECT * FROM read_parquet('{dedup_path}')
  WHERE snapshot_date >= DATE '2024-07-01'
) TO '{test_path}' (FORMAT PARQUET);
""")

split_stats = con.execute(f"""
WITH t AS (
  SELECT 'train' AS split, * FROM read_parquet('{train_path}')
  UNION ALL
  SELECT 'val' AS split, * FROM read_parquet('{val_path}')
  UNION ALL
  SELECT 'test' AS split, * FROM read_parquet('{test_path}')
)
SELECT split, COUNT(*) AS n_rows, AVG(target_30d) AS target_rate,
       MIN(snapshot_date) AS min_date, MAX(snapshot_date) AS max_date
FROM t
GROUP BY split
ORDER BY split
""").fetchdf()

split_stats

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,n_rows,target_rate,min_date,max_date
0,test,261737,0.034290,2024-07-01,2024-09-01
1,train,2974001,0.038784,2022-08-01,2024-03-01
2,val,296982,0.047202,2024-04-01,2024-06-01


Load splits into pandas

In [15]:
train_df = pd.read_parquet(train_path)
val_df   = pd.read_parquet(val_path)
test_df  = pd.read_parquet(test_path)

cat_cols = ["contract_status", "technology", "sales_channel", "crm_package_name"]
for df in [train_df, val_df, test_df]:
    for c in cat_cols:
        df[c] = df[c].astype(str).fillna("Unknown").str.strip()

train_df.shape, val_df.shape, test_df.shape

((2974001, 28), (296982, 28), (261737, 28))

Add log1p features

In [16]:
def add_log_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["sum_download_30d", "sum_upload_30d", "avg_talk_time_30d", "avg_hold_time_30d"]:
        if col in df.columns:
            df[f"log1p_{col}"] = np.log1p(df[col].clip(lower=0))
    return df

train_df = add_log_features(train_df)
val_df   = add_log_features(val_df)
test_df  = add_log_features(test_df)

Define features/target

In [17]:
TARGET = "target_30d"
ID_COL = "unique_customer_identifier"
DATE_COL = "snapshot_date"

feature_cols = [c for c in train_df.columns if c not in [TARGET, ID_COL, DATE_COL]]

X_train, y_train = train_df[feature_cols], train_df[TARGET].astype(int)
X_val, y_val     = val_df[feature_cols], val_df[TARGET].astype(int)
X_test, y_test   = test_df[feature_cols], test_df[TARGET].astype(int)

# detect categoricals dynamically (but keep our known)
cat_cols = train_df[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]

len(feature_cols), len(cat_cols), len(num_cols), cat_cols

(29,
 4,
 25,
 ['contract_status', 'technology', 'sales_channel', 'crm_package_name'])

Logistic Regression baseline Model

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

prep_lr = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

lr_model = Pipeline([
    ("prep", prep_lr),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

lr_model.fit(X_train, y_train)
val_proba_lr = lr_model.predict_proba(X_val)[:, 1]

lr_val_roc = roc_auc_score(y_val, val_proba_lr)
lr_val_pr  = average_precision_score(y_val, val_proba_lr)

print("LR Val ROC-AUC:", lr_val_roc)
print("LR Val PR-AUC :", lr_val_pr)

LR Val ROC-AUC: 0.9014161857206539
LR Val PR-AUC : 0.34753795173407714


HistGradientBoosting Model

In [19]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

numeric_pipe_hgb = Pipeline([("imputer", SimpleImputer(strategy="median"))])

categorical_pipe_hgb = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

prep_hgb = ColumnTransformer([
    ("num", numeric_pipe_hgb, num_cols),
    ("cat", categorical_pipe_hgb, cat_cols),
])

hgb_model = Pipeline([
    ("prep", prep_hgb),
    ("model", HistGradientBoostingClassifier(
        max_depth=6,
        learning_rate=0.08,
        max_iter=300,
        random_state=42
    ))
])

sw_train = compute_sample_weight(class_weight="balanced", y=y_train)
hgb_model.fit(X_train, y_train, model__sample_weight=sw_train)

val_proba_hgb = hgb_model.predict_proba(X_val)[:, 1]
hgb_val_roc = roc_auc_score(y_val, val_proba_hgb)
hgb_val_pr  = average_precision_score(y_val, val_proba_hgb)

print("HGB Val ROC-AUC:", hgb_val_roc)
print("HGB Val PR-AUC :", hgb_val_pr)

HGB Val ROC-AUC: 0.9181142755964318
HGB Val PR-AUC : 0.4012745052695745


Select Best Model

In [20]:
best_model = hgb_model if hgb_val_pr >= lr_val_pr else lr_model
best_name = "HGB" if best_model is hgb_model else "LR_scaled"

test_proba = best_model.predict_proba(X_test)[:, 1]
test_roc = roc_auc_score(y_test, test_proba)
test_pr  = average_precision_score(y_test, test_proba)

print("Winner:", best_name)
print("TEST ROC-AUC:", test_roc)
print("TEST PR-AUC :", test_pr)

Winner: HGB
TEST ROC-AUC: 0.9096488069997651
TEST PR-AUC : 0.29714652720346174


Create categorical levels for Streamlit

In [21]:
categorical_levels = {}
for c in cat_cols:
    top_vals = (
        train_df[c].astype(str).fillna("Unknown").str.strip()
        .value_counts()
        .head(50)
        .index
        .tolist()
    )
    if "Unknown" not in top_vals:
        top_vals = ["Unknown"] + top_vals
    categorical_levels[c] = top_vals

categorical_levels["contract_status"][:15]

['Unknown',
 '02 In Contract',
 '06 OOC',
 '03 Soon to be OOC',
 '01 Early Contract',
 '05 Newly OOC',
 '04 Coming OOC']

Feature importance

In [23]:
from sklearn.inspection import permutation_importance
import pandas as pd

n_sample = min(50000, len(val_df))
val_sample = val_df.sample(n=n_sample, random_state=42)

perm = permutation_importance(
    best_model,
    val_sample[feature_cols],
    val_sample[TARGET].astype(int),
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

importances = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)

# UI shouldn't ask user for derived log features (we compute them automatically)
ui_features = [f for f in importances.index if not f.startswith("log1p_")]

# ui_features is already a list, so just slice
top_features_for_ui = ui_features[:12]

display(importances.head(15))
top_features_for_ui

ooc_days               0.134858
contract_dd_cancels    0.077160
dd_cancel_60_day       0.047079
line_speed             0.038902
tenure_days            0.034435
speed                  0.028198
avg_download_30d       0.018768
calls_Loyalty_30d      0.015125
sum_download_30d       0.014815
avg_upload_30d         0.013192
sales_channel          0.012890
calls_30d              0.007705
avg_talk_time_30d      0.006250
sum_upload_30d         0.006064
contract_status        0.005764
dtype: float64

['ooc_days',
 'contract_dd_cancels',
 'dd_cancel_60_day',
 'line_speed',
 'tenure_days',
 'speed',
 'avg_download_30d',
 'calls_Loyalty_30d',
 'sum_download_30d',
 'avg_upload_30d',
 'sales_channel',
 'calls_30d']

Save model and metadata

In [24]:
model_file = MODELS_DIR / "cease_risk_model.joblib"
meta_file  = MODELS_DIR / "metadata.json"

joblib.dump(best_model, model_file)

metadata = {
    "trained_at": datetime.utcnow().isoformat(),
    "model_name": "HistGradientBoostingClassifier" if best_name == "HGB" else "LogisticRegression_scaled",
    "lookback_days": LOOKBACK_DAYS,
    "horizon_days": HORIZON_DAYS,
    "val_roc_auc_lr": float(lr_val_roc),
    "val_pr_auc_lr": float(lr_val_pr),
    "val_roc_auc_hgb": float(hgb_val_roc),
    "val_pr_auc_hgb": float(hgb_val_pr),
    "test_roc_auc": float(test_roc),
    "test_pr_auc": float(test_pr),
    "features": feature_cols,
    "categorical_features": cat_cols,
    "numeric_features": num_cols,
    "categorical_levels": categorical_levels,
    "feature_importance": {k: float(v) for k, v in importances.items()},
    "top_features_for_ui": top_features_for_ui,
    "top_call_types": TOP_CALL_TYPES,
}

with open(meta_file, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved model:", model_file, "size(MB):", round(model_file.stat().st_size/1e6, 2))
print("Saved metadata:", meta_file)

Saved model: c:\Users\icons\Documents\UKTelecomCeasePrediction\models\cease_risk_model.joblib size(MB): 0.84
Saved metadata: c:\Users\icons\Documents\UKTelecomCeasePrediction\models\metadata.json
